# 🔬 AfriOmics | Metatranscriptomics Module
## Step-by-step Jupyter Notebook Workflow
---
**What this notebook does:**
1. Remove ribosomal RNA from reads
2. Quantify active gene expression from microbial communities
3. Profile active metabolic pathways (HUMAnN3)
4. Identify differentially expressed genes between disease groups
5. Map to KEGG/MetaCyc pathways
6. Compute DNA vs RNA ratio (potential vs active)

> 💡 **Key concept:** Metagenomics tells you WHO is there. Metatranscriptomics tells you what they are **actively doing** at the moment of sampling.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pandas', 'numpy', 'matplotlib', 'seaborn',
                'plotly', 'scipy', 'scikit-learn'], check=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

print('✅ AfriOmics Metatranscriptomics Notebook v1.0')

## ⚙️ Configuration

In [ ]:
CONFIG = {
    'results_dir'    : 'results/metatranscriptomics/',
    'metadata_file'  : 'config/samples.tsv',
    'sortmerna_db'   : 'resources/sortmerna_db/smr_v4.3_sensitive_db.fasta',
    'star_index'     : 'resources/star_index/',
    'group_col'      : 'disease',
    'ref_level'      : 'healthy',
    'padj_cutoff'    : 0.05,
    'lfc_cutoff'     : 1.0,
    'threads'        : 8,
}

for d in ['rrna_removed', 'aligned', 'counts', 'humann3', 'deseq2', 'pathway_enrichment', 'dna_rna_ratio']:
    Path(f"{CONFIG['results_dir']}/{d}").mkdir(parents=True, exist_ok=True)

print('✅ Configuration set.')

---
## STEP 1 — rRNA Removal with SortMeRNA
**Why:** Ribosomal RNA reads dominate RNA-seq libraries (up to 90%) and carry no coding information.

**Tool:** SortMeRNA uses profile hidden Markov models of SILVA + RFAM rRNA databases.

**Output:** Two sets of reads — rRNA (discarded) and non-rRNA (mRNA — the ones we want).

In [ ]:
import subprocess

def run_sortmerna(sample_id, r1_path, r2_path, ref_db, outdir, threads=8):
    """
    Remove rRNA reads using SortMeRNA.
    Returns paths to cleaned (mRNA) reads.
    """
    from pathlib import Path
    outdir  = Path(outdir)
    workdir = outdir / f'{sample_id}_tmp'
    workdir.mkdir(parents=True, exist_ok=True)

    r1_clean = outdir / f'{sample_id}_R1.fastq.gz'
    r2_clean = outdir / f'{sample_id}_R2.fastq.gz'

    cmd = [
        'sortmerna',
        '--ref',      ref_db,
        '--reads',    r1_path,
        '--reads',    r2_path,
        '--paired-in',
        '--out2',
        '--aligned',  str(workdir / 'rrna'),
        '--other',    str(workdir / 'nonrrna'),
        '--threads',  str(threads),
        '--fastx', '--zip-out',
        '--workdir',  str(workdir)
    ]

    print(f'  Removing rRNA from {sample_id}...')
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode == 0:
        # Move outputs
        (workdir / 'nonrrna_fwd.fastq.gz').rename(r1_clean)
        (workdir / 'nonrrna_rev.fastq.gz').rename(r2_clean)
        print(f'  ✅ {sample_id}: rRNA removed')
        return str(r1_clean), str(r2_clean)
    else:
        print(f'  ❌ SortMeRNA failed: {result.stderr[:300]}')
        return None, None


def compute_rrna_fraction(original_r1, cleaned_r1):
    """Estimate the fraction of reads that were rRNA."""
    def count_reads(f):
        result = subprocess.run(f'zcat {f} | awk "NR%4==1" | wc -l',
                                shell=True, capture_output=True, text=True)
        return int(result.stdout.strip())

    total   = count_reads(original_r1)
    cleaned = count_reads(cleaned_r1)
    rrna_pct = round((total - cleaned) / total * 100, 1) if total > 0 else 0
    return total, cleaned, rrna_pct

print('✅ rRNA removal functions ready.')

---
## STEP 2 — DNA vs RNA Ratio Analysis
**What:** Compares what microbes can do (DNA/metagenomics) vs what they are doing (RNA/metatranscriptomics).

**Why:** A pathway encoded in the genome but not expressed may be dormant. A highly expressed pathway not well-represented in DNA suggests a highly active but low-abundance organism.

**This is one of the most powerful outputs of a combined metagenomics + metatranscriptomics study.**

In [ ]:
def compute_dna_rna_ratio(dna_pathway_df, rna_pathway_df, metadata_df,
                           pseudocount=1e-6):
    """
    Compute DNA:RNA ratio per pathway per sample.

    Parameters:
    -----------
    dna_pathway_df : pathways × samples (from metagenomics HUMAnN3)
    rna_pathway_df : pathways × samples (from metatranscriptomics HUMAnN3)
    metadata_df    : sample metadata

    Returns:
    --------
    ratio_df     : log2(RNA/DNA) per pathway per sample
    summary_df   : mean ratio per pathway per disease group
    """
    # Find shared pathways and samples
    shared_paths   = list(set(dna_pathway_df.index) & set(rna_pathway_df.index))
    shared_samples = list(set(dna_pathway_df.columns) & set(rna_pathway_df.columns)
                          & set(metadata_df.index))

    print(f'Shared pathways : {len(shared_paths)}')
    print(f'Shared samples  : {len(shared_samples)}')

    dna = dna_pathway_df.loc[shared_paths, shared_samples].copy()
    rna = rna_pathway_df.loc[shared_paths, shared_samples].copy()

    # log2(RNA/DNA) ratio — positive = more active than encoded potential
    ratio_df = np.log2((rna + pseudocount) / (dna + pseudocount))

    # Classify pathways by activity pattern
    mean_ratio = ratio_df.mean(axis=1)
    pathway_class = pd.Series(index=mean_ratio.index, dtype=str)
    pathway_class[mean_ratio >  1.0] = 'Highly_Active'     # RNA >> DNA
    pathway_class[mean_ratio <  -1.0] = 'Dormant'          # DNA >> RNA
    pathway_class[(mean_ratio >= -1.0) & (mean_ratio <= 1.0)] = 'Constitutive'  # RNA ≈ DNA

    print(f'\nPathway Activity Classes:')
    print(pathway_class.value_counts().to_string())

    # Summary by disease group
    meta_sub = metadata_df.loc[shared_samples]
    groups   = meta_sub['disease'].unique() if 'disease' in meta_sub.columns else ['all']

    summary_rows = []
    for pathway in shared_paths:
        row = {'pathway': pathway, 'activity_class': pathway_class[pathway], 'mean_overall': mean_ratio[pathway]}
        for grp in groups:
            grp_samples = meta_sub[meta_sub['disease'] == grp].index.tolist() if 'disease' in meta_sub.columns else shared_samples
            grp_samples = [s for s in grp_samples if s in ratio_df.columns]
            if grp_samples:
                row[f'mean_{grp}'] = ratio_df.loc[pathway, grp_samples].mean()
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows).sort_values('mean_overall', ascending=False)

    return ratio_df, summary_df, pathway_class


def plot_dna_rna_ratio(summary_df, top_n=25):
    """Bar chart of top differentially active pathways (RNA vs DNA)."""
    top_active  = summary_df.nlargest(top_n // 2, 'mean_overall')
    top_dormant = summary_df.nsmallest(top_n // 2, 'mean_overall')
    plot_df = pd.concat([top_active, top_dormant])

    colour_map = {'Highly_Active': '#4aff91', 'Constitutive': '#5bc4ff', 'Dormant': '#ff6b6b'}
    colours = [colour_map.get(c, '#7a9b7e') for c in plot_df['activity_class']]

    fig = px.bar(
        plot_df.sort_values('mean_overall'),
        x     = 'mean_overall',
        y     = 'pathway',
        color = 'activity_class',
        color_discrete_map = colour_map,
        orientation = 'h',
        title = 'DNA vs RNA Ratio per Pathway<br><sup>Positive = more active than encoded; Negative = dormant</sup>',
        labels = {'mean_overall': 'log2(RNA/DNA)', 'pathway': 'Pathway', 'activity_class': 'Activity'},
        template = 'plotly_dark'
    )
    fig.add_vline(x=0, line_color='#f5c842', line_dash='dash')
    fig.add_vline(x=1, line_color='#4aff91', line_dash='dot', opacity=0.5)
    fig.add_vline(x=-1, line_color='#ff6b6b', line_dash='dot', opacity=0.5)
    fig.update_layout(paper_bgcolor='#111710', plot_bgcolor='#111710', font_color='#e8f0e9',
                      height=max(400, len(plot_df) * 22))
    return fig

print('✅ DNA vs RNA ratio functions ready.')
print('\nKey insight: A pathway with log2(RNA/DNA) > 1 means the microbes are')
print('more transcriptionally active than their genome abundance would predict.')

---
## ▶ DEMO — Run on Synthetic Data

In [ ]:
np.random.seed(42)
n_samples  = 12
n_pathways = 60

pathway_names = [
    'PWY-5667: CDP-diacylglycerol biosynthesis',
    'BUTANAL-OXIDATION-PWY: butanoate oxidation',
    'PWY-3781: aerobic respiration',
    'GLYCOLYSIS: glycolysis',
    'TCA: TCA cycle',
    'FASYN-ELONG-PWY: fatty acid elongation',
    'PWY-5695: butyrate biosynthesis I',
    'ARGININE-SYN4-PWY: arginine biosynthesis',
    'TRPSYN-PWY: tryptophan biosynthesis',
    'PWY-6527: stachyose degradation',
    'GALLATE-DEGRADATION: gallate degradation',
    'SHIKIMATE-PWY: shikimate pathway',
] + [f'Pathway_{i}' for i in range(n_pathways - 12)]

sample_names = [f'AFRI_{str(i).zfill(3)}' for i in range(1, n_samples + 1)]

# DNA abundances (metagenomics)
dna_df = pd.DataFrame(
    np.random.lognormal(mean=2, sigma=1, size=(n_pathways, n_samples)),
    index=pathway_names, columns=sample_names
)

# RNA abundances (metatranscriptomics) — some pathways highly upregulated in disease
rna_df = dna_df.copy()
disease_samples = sample_names[:4]  # First 4 are 'malaria'
# Butyrate biosynthesis is downregulated in malaria patients
rna_df.loc['PWY-5695: butyrate biosynthesis I', disease_samples] *= 0.1
# Tryptophan pathway is upregulated
rna_df.loc['TRPSYN-PWY: tryptophan biosynthesis', disease_samples] *= 5.0
# Add noise
rna_df *= np.random.lognormal(0, 0.3, rna_df.shape)

metadata_demo = pd.DataFrame({
    'disease'   : ['malaria'] * 4 + ['HIV'] * 4 + ['healthy'] * 4,
    'body_site' : ['gut'] * n_samples,
    'region'    : ['west_africa'] * 4 + ['east_africa'] * 4 + ['southern_africa'] * 4
}, index=sample_names)

print('Demo data generated:')
print(f'  {n_pathways} pathways × {n_samples} samples')
print(f'  Groups: {metadata_demo["disease"].value_counts().to_dict()}')

# Run DNA vs RNA ratio
ratio_df, summary_df, pathway_class = compute_dna_rna_ratio(dna_df, rna_df, metadata_demo)

print('\nTop 5 most active pathways (RNA >> DNA):')
print(summary_df.head()[['pathway', 'activity_class', 'mean_overall']].to_string(index=False))

fig = plot_dna_rna_ratio(summary_df, top_n=20)
fig.show()

# Save
ratio_df.to_csv(f"{CONFIG['results_dir']}/dna_rna_ratio/dna_rna_ratio.tsv", sep='\t')
summary_df.to_csv(f"{CONFIG['results_dir']}/dna_rna_ratio/dna_rna_summary.tsv", sep='\t', index=False)
print('\n✅ Metatranscriptomics demo complete.')